# 模块概述

WtExecMon 是 WonderTrader 执行监控模块，负责管理执行器、交易通道、行情通道等组件的生命周期，协调各组件协同工作，实现智能订单执行。主要包括：
- 执行器运行器核心管理
- 交易通道和行情通道管理
- 执行器工厂和执行器管理
- 数据管理和K线数据管理
- 仓位管理和目标仓位提交
- C接口导出供外部调用

1. **运行器层**（WtExecRunner）：
   - 核心运行器，管理执行器、交易通道、行情通道等组件的生命周期
   - 实现 IParserStub 接口，接收解析器推送的行情数据
   - 实现 IExecuterStub 接口，为执行器提供基础信息查询功能
   - 协调各组件协同工作，实现智能订单执行

2. **适配器管理层**（TraderAdapterMgr + ParserAdapterMgr）：
   - **TraderAdapterMgr**：交易适配器管理器，管理多个交易通道
   - **ParserAdapterMgr**：解析器适配器管理器，管理多个行情通道
   - 支持多通道配置和动态加载

3. **执行器管理层**（WtExecuterFactory + WtExecuterMgr）：
   - **WtExecuterFactory**：执行器工厂，创建和管理执行器实例
   - **WtExecuterMgr**：执行器管理器，管理执行器的运行
   - 支持本地执行器、差分执行器、分布式执行器等类型

4. **数据管理层**（WtSimpDataMgr + WTSBaseDataMgr + WTSHotMgr）：
   - **WtSimpDataMgr**：简单数据管理器，管理行情数据和K线数据
   - **WTSBaseDataMgr**：基础数据管理器，管理商品、合约、交易时段等基础数据
   - **WTSHotMgr**：热点合约管理器，管理主力合约、次主力合约等

5. **工具支持层**（ActionPolicyMgr + WtExecPorter）：
   - **ActionPolicyMgr**：开平策略管理器，管理开仓和平仓策略
   - **WtExecPorter**：C接口导出，提供跨语言调用接口

6. **数据流程**：
   - 行情数据：解析器 → ParserAdapter → WtExecRunner → WtSimpDataMgr → 执行器
   - 交易指令：执行器 → TraderAdapter → 交易通道
   - 目标仓位：外部调用 → WtExecRunner → WtExecuterMgr → 执行器

# 层次关系图
```mermaid
graph LR
    %% 样式定义
    classDef runnerClass fill:#e1f5ff,stroke:#01579b,stroke-width:3px,color:#000;
    classDef adapterClass fill:#f3e5f5,stroke:#4a148c,stroke-width:2px,color:#000;
    classDef executerClass fill:#fff3e0,stroke:#e65100,stroke-width:2px,color:#000;
    classDef dataClass fill:#e8f5e9,stroke:#1b5e20,stroke-width:2px,color:#000;
    classDef utilClass fill:#ffe0b2,stroke:#e65100,stroke-width:2px,color:#000;
    classDef interfaceClass fill:#f5f5f5,stroke:#616161,stroke-width:1px,stroke-dasharray: 5 5,color:#000;
    classDef porterClass fill:#fce4ec,stroke:#880e4f,stroke-width:2px,color:#000;

    %% 接口层
    subgraph Interfaces["接口层 - 抽象接口"]
        direction TB
        IParserStub["IParserStub<br/>解析器存根接口<br/>• 行情数据推送<br/>• 订单队列推送<br/>• 订单明细推送<br/>• 成交明细推送"]:::interfaceClass
        IExecuterStub["IExecuterStub<br/>执行器存根接口<br/>• 实时时间查询<br/>• 商品信息查询<br/>• 交易会话查询<br/>• 热点合约查询<br/>• 交易日期查询"]:::interfaceClass
        IDataManager["IDataManager<br/>数据管理器接口<br/>• Tick切片查询<br/>• K线切片查询<br/>• 最新Tick查询"]:::interfaceClass
        IDataReader["IDataReader<br/>数据读取器接口<br/>• 历史Tick读取<br/>• 历史K线读取"]:::interfaceClass
    end

    %% 运行器层
    subgraph Runner["运行器层 - 核心运行器"]
        direction TB
        WtExecRunner["WtExecRunner<br/>执行器运行器<br/>• 组件生命周期管理<br/>• 行情数据处理<br/>• 执行器支持<br/>• 仓位管理<br/>• 数据管理"]:::runnerClass
    end

    %% 适配器管理层
    subgraph Adapters["适配器管理层 - 通道管理"]
        direction TB
        TraderAdapterMgr["TraderAdapterMgr<br/>交易适配器管理器<br/>• 管理多个交易通道<br/>• 交易指令转发<br/>• 交易回报处理"]:::adapterClass
        ParserAdapterMgr["ParserAdapterMgr<br/>解析器适配器管理器<br/>• 管理多个行情通道<br/>• 行情数据转发<br/>• 数据过滤和标准化"]:::adapterClass
    end

    %% 执行器管理层
    subgraph Executers["执行器管理层 - 执行器管理"]
        direction TB
        WtExecuterFactory["WtExecuterFactory<br/>执行器工厂<br/>• 创建执行器实例<br/>• 加载执行器工厂<br/>• 管理执行器类型"]:::executerClass
        WtExecuterMgr["WtExecuterMgr<br/>执行器管理器<br/>• 管理执行器运行<br/>• 处理行情数据<br/>• 管理目标仓位"]:::executerClass
    end

    %% 数据管理层
    subgraph DataLayer["数据管理层 - 数据管理"]
        direction TB
        WtSimpDataMgr["WtSimpDataMgr<br/>简单数据管理器<br/>• 实时Tick缓存<br/>• K线数据缓存<br/>• 历史数据读取<br/>• 时间管理"]:::dataClass
        WTSBaseDataMgr["WTSBaseDataMgr<br/>基础数据管理器<br/>• 商品信息管理<br/>• 合约信息管理<br/>• 交易时段管理<br/>• 节假日管理"]:::dataClass
        WTSHotMgr["WTSHotMgr<br/>热点合约管理器<br/>• 主力合约管理<br/>• 次主力合约管理<br/>• 合约切换管理"]:::dataClass
    end

    %% 工具支持层
    subgraph Utils["工具支持层 - 辅助功能"]
        direction TB
        ActionPolicyMgr["ActionPolicyMgr<br/>开平策略管理器<br/>• 开仓策略管理<br/>• 平仓策略管理<br/>• 品种规则映射"]:::utilClass
        WtExecPorter["WtExecPorter<br/>C接口导出<br/>• 模块初始化<br/>• 配置加载<br/>• 运行控制<br/>• 仓位管理<br/>• 日志记录"]:::porterClass
    end

    %% 继承关系
    WtExecRunner -.->|"实现"| IParserStub
    WtExecRunner -.->|"实现"| IExecuterStub
    WtSimpDataMgr -.->|"实现"| IDataManager
    WtSimpDataMgr -.->|"实现"| IDataReaderSink

    %% 运行器组合关系
    WtExecRunner -->|"包含"| TraderAdapterMgr
    WtExecRunner -->|"包含"| ParserAdapterMgr
    WtExecRunner -->|"包含"| WtExecuterFactory
    WtExecRunner -->|"包含"| WtExecuterMgr
    WtExecRunner -->|"包含"| WtSimpDataMgr
    WtExecRunner -->|"包含"| WTSBaseDataMgr
    WtExecRunner -->|"包含"| WTSHotMgr
    WtExecRunner -->|"包含"| ActionPolicyMgr

    %% 数据流关系
    ParserAdapterMgr -->|"推送行情"| WtExecRunner
    WtExecRunner -->|"更新数据"| WtSimpDataMgr
    WtExecRunner -->|"转发行情"| WtExecuterMgr
    WtExecRunner -->|"查询信息"| WTSBaseDataMgr
    WtExecRunner -->|"查询热点"| WTSHotMgr

    %% 执行器关系
    WtExecuterFactory -->|"创建"| WtExecuterMgr
    WtExecuterMgr -->|"使用"| TraderAdapterMgr
    WtExecuterMgr -->|"使用"| WtSimpDataMgr

    %% 数据管理器关系
    WtSimpDataMgr -->|"使用"| IDataReader
    WtSimpDataMgr -->|"查询基础数据"| WTSBaseDataMgr
    WtSimpDataMgr -->|"查询热点"| WTSHotMgr

    %% C接口关系
    WtExecPorter -.->|"调用"| WtExecRunner

    %% 应用样式
    class WtExecRunner runnerClass
    class TraderAdapterMgr,ParserAdapterMgr adapterClass
    class WtExecuterFactory,WtExecuterMgr executerClass
    class WtSimpDataMgr,WTSBaseDataMgr,WTSHotMgr dataClass
    class ActionPolicyMgr utilClass
    class WtExecPorter porterClass
    class IParserStub,IExecuterStub,IDataManager,IDataReader interfaceClass
```